# 정답 노트북\n\n코랩에서 실행하세요. 데이터가 없으면 자동으로 crack.zip을 받습니다.\n

In [ ]:
try:\n    from google.colab import drive\n    drive.mount('/content/gdrive')\nexcept Exception as e:\n    print('skip drive', e)\n

In [ ]:
import os\nfrom pathlib import Path\nimport urllib.request, zipfile\n\ncandidates = [\n    Path('/content/gdrive/MyDrive/data/crack'),\n    Path('/content/gdrive/My Drive/data/crack'),\n    Path('/content/data/crack'),\n    Path('crack'),\n]\ndata_dir = next((p for p in candidates if (p/'train').is_dir() and (p/'test').is_dir()), None)\nif data_dir is None:\n    zip_path = Path('/content/crack.zip') if Path('/content').exists() else Path('crack.zip')\n    extract_to = Path('/content/data/crack') if Path('/content').exists() else Path('crack')\n    extract_to.mkdir(parents=True, exist_ok=True)\n    urllib.request.urlretrieve('https://bit.ly/crack_dataset', zip_path)\n    with zipfile.ZipFile(zip_path) as zf:\n        zf.extractall(extract_to)\n    if not (extract_to/'train').is_dir():\n        for child in extract_to.rglob('train'):\n            if child.is_dir() and (child.parent/'test').is_dir():\n                extract_to = child.parent\n                break\n    data_dir = extract_to\nos.chdir(data_dir)\nprint(os.getcwd(), Path('train').is_dir(), Path('test').is_dir())\n

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator\ntrain_datagen = ImageDataGenerator(rescale=1./255)\ntraining_set = train_datagen.flow_from_directory('train', target_size=(64,64), batch_size=32, shuffle=True, class_mode='categorical')\ntest_datagen = ImageDataGenerator(rescale=1./255)\ntest_set = test_datagen.flow_from_directory('test', target_size=(64,64), shuffle=False, class_mode='categorical')\n

In [ ]:
from tensorflow.keras.applications.vgg16 import VGG16\nfrom tensorflow.keras.models import Sequential\nfrom tensorflow.keras.layers import Dense, Flatten\nvgg = VGG16(include_top=False, weights='imagenet', input_shape=(64,64,3))\nvgg.trainable = False\nmodel = Sequential([vgg, Flatten(), Dense(64, activation='relu'), Dense(2, activation='softmax')])\nmodel.summary()\n

In [ ]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])\nmodel.fit(training_set, epochs=5)\n

In [ ]:
print(model.evaluate(test_set))\npred = model.predict(test_set)\nprint(training_set.class_indices)\n

In [ ]:
import numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nfrom sklearn.metrics import confusion_matrix\nPredicted = np.argmax(pred, axis=1)\nActual = test_set.labels\nconf = confusion_matrix(Actual, Predicted)\nsns.heatmap(conf, annot=True, cmap='BuPu', fmt='d')\nplt.title('Crack Classification')\nplt.xlabel('Predicted')\nplt.ylabel('Actual')\nplt.show()\n